In [1]:
from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd
fileType = 'band'
bandNames = {'B3', 'B4', 'B5', 'ST_B10'}
includeMetadata = True

Directory 'Unprocessed' already exists.
Directory 'utils' already exists.
Logging in...


Login Successful, API Key Received!


In [2]:
# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_San_Antonio_TX.shp"
city = shapefile.replace('Polygon_', '').replace('.shp', '')
aoi_geodf = gpd.read_file(shapefile_folder + shapefile)
aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
if aoi_geodf.empty:
    sys.exit("Error: Shapefile contains no data.")
print("Shapefile loaded successfully.")

Shapefile loaded successfully.


In [3]:
centroid = aoi_geodf.geometry.centroid.iloc[0]
m = folium.Map(
    location=[centroid.y, centroid.x], 
    zoom_start=9, tiles="openstreetmap", width="100%", height="100%", attributionControl=0
)
# Cell 4: Add Polygon to Map
folium.GeoJson(aoi_geodf).add_to(m)
m

C:\Users\jesse\AppData\Local\Temp\ipykernel_12628\3196744355.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = aoi_geodf.geometry.centroid.iloc[0]


In [6]:
# Cell 5: Define Scene Search Parameters, temporal is inclusive
datasetName = 'landsat_ot_c2_l2'
spatialFilter = {
    'filterType': 'mbr',
    'lowerLeft': {
        'latitude': aoi_geodf.geometry.bounds.miny[0],
        'longitude': aoi_geodf.geometry.bounds.minx[0]
    },
    'upperRight': {
        'latitude': aoi_geodf.geometry.bounds.maxy[0],
        'longitude': aoi_geodf.geometry.bounds.maxx[0]
    }
}

temporalFilter = {'start': '2020-04-12', 'end': '2020-04-15'}
cloudCoverFilter = {'min': 0, 'max': 20}
search_payload = {
    'datasetName': datasetName,
    'sceneFilter': {
        'spatialFilter': spatialFilter,
        'acquisitionFilter': temporalFilter,
        'cloudCoverFilter': cloudCoverFilter
    }
}

In [7]:
# Cell 6: Search for Scenes
scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
pd.json_normalize(scenes['results'])

,browse,cloudCover,entityId,displayId,orderingId,metadata,hasCustomizedMetadata,publishDate,options.bulk,options.download,...,options.secondary,selected.bulk,selected.compare,selected.order,spatialBounds.type,spatialBounds.coordinates,spatialCoverage.type,spatialCoverage.coordinates,temporalCoverage.endDate,temporalCoverage.startDate
0,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",0,LC80270392020103LGN00,LC08_L2SP_027039_20200412_20200822_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-99.04812, 29.24596], [-99.04812, 31.3539],...",Polygon,"[[[-99.04812, 29.6275], [-97.13325, 29.24596],...",2020-04-12 00:00:00,2020-04-12 00:00:00
1,"[{'id': '5fb4ba12d7ec307f', 'browseRotationEna...",1,LC80270402020103LGN00,LC08_L2SP_027040_20200412_20200822_02_T1,None,"[{'id': '5e83d1508031a4a3', 'fieldName': 'ID',...",None,2022-06-22 18:27:47-05,True,True,...,False,False,False,False,Polygon,"[[[-99.40271, 27.80988], [-99.40271, 29.91645]...",Polygon,"[[[-99.40271, 28.189], [-97.51381, 27.80988], ...",2020-04-12 00:00:00,2020-04-12 00:00:00


In [8]:
# Cell 7: Collect Entity IDs
entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]

In [9]:
# Cell 8: Prepare Scene List for Download
listId = f"temp_{datasetName}_list"
scn_list_add_payload = {
    "listId": listId,
    'idField': 'entityId',
    "entityIds": entityIds,
    "datasetName": datasetName
}
sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)

2

In [10]:
# Cell 9: Prepare Download Options
download_opt_payload = {
    "listId": listId,
    "datasetName": datasetName
}
products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
pd.json_normalize(products)


,id,downloadName,displayId,entityId,datasetId,available,filesize,productName,productCode,bulkAvailable,downloadSystem,secondaryDownloads,fileGroups
0,5e83d14fec7cae84,None,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,1008269944,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
1,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
3,632210d4770592cf,None,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,False,1008269944,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
4,5e83d14fec7cae84,None,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,983572232,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
5,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
6,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
7,632210d4770592cf,None,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,False,983572232,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None


In [11]:
# Cell 10: Collect Files to Download
downloads = []
for product in products:
    if product["secondaryDownloads"]:
        for secDownload in product["secondaryDownloads"]:
            if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
            if includeMetadata and secDownload['displayId'].endswith('_MTL.txt'):
                downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})


In [12]:
# Cell 11: Submit Download Request
download_req_payload = {
    "downloads": downloads,
    "label": listId
}
download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)


In [20]:
# Cell 12: Download Files
for result in download_request_results['availableDownloads']:
    runDownload(threads, result['url'])

{'downloadId': 712979955, 'eulaCode': None, 'entityId': 'L2ST_LC08_L2SP_027039_20200412_20200822_02_T1_MTL_TXT', 'url': 'https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2020/027/039/LC08_L2SP_027039_20200412_20200822_02_T1/LC08_L2SP_027039_20200412_20200822_02_T1_MTL.txt?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6MjczNjI1NzcsImRvd25sb2FkSWQiOjcxMjk3OTk1NSwiZGF0ZUdlbmVyYXRlZCI6IjIwMjQtMTItMTZUMTE6NDQ6MTYtMDY6MDAiLCJpZCI6IkxDMDhfTDJTUF8wMjcwMzlfMjAyMDA0MTJfMjAyMDA4MjJfMDJfVDFfTVRMLnR4dCIsInNpZ25hdHVyZSI6IiQ1JCRicFRlOU1xNnJXUnpPZlh3cHU5ajI2a1VcL21yWkFnTm1ISGN3akVxa2ZuMiJ9'}
MTL
2020-04-12
{'downloadId': 712979956, 'eulaCode': None, 'entityId': 'L2SR_LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3_TIF', 'url': 'https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2020/027/039/LC08_L2SP_027039_20200412_20200822_02_T1/LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3.TIF?requestSignature=eyJkb3dubG9hZEFwcCI6Ik0yTSIsImNvbnRhY3RJZCI6Mj

In [13]:
# Cell 13: Clean Up
remove_scnlst_payload = {"listId": listId}
sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)

In [6]:
# Cell 15: Verify Downloads
import rasterio
from shapely.geometry import box

usableTIFFs = []
print(os.listdir(unprocessed_dir))
# Ensure the shapefile matches the TIF CRS
for tif_file in os.listdir(unprocessed_dir):
    if tif_file.endswith(".TIF"):
        tif_path = os.path.join(unprocessed_dir, tif_file)
    with rasterio.open(tif_path) as src:
        # Reproject shapefile to match TIF file CRS
        aoi_geodf_proj = aoi_geodf.to_crs(src.crs)
        tif_bounds = box(*src.bounds)
        
        # Check containment
        for idx, geom in enumerate(aoi_geodf_proj.geometry):
            if tif_bounds.contains(geom):
                usableTIFFs.append(tif_file)
                print(f"Polygon {city} is fully inside {tif_file}")
            else:
                print(f"Polygon {city} is NOT fully inside {tif_file}")
print(usableTIFFs)

['LC08_L2SP_027039_20200412_20200822_02_T1_MTL.txt', 'LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF', 'LC08_L2SP_027040_20200412_20200822_02_T1_MTL.txt', 'LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF']
Polygon 0 is NOT fully inside LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF
Polygon 0 is fully inside LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF
['LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF']


In [29]:
clipUnprocessedRasters(usableTIFFs, aoi_geodf_proj)
for file in os.listdir(unprocessed_dir):
    if ".txt" in file or "Clipped_" in file:
        date, band, city = getMetaFromLandsatTIRs(file), city
        if band is 'B10':
            moveToRaw(file, 'LST', date, city)
        if band is 'B3':
            moveToRaw(file, 'NDWI', date, city)
        if band is 'B4':
            moveToRaw(file, 'NDVI', date, city)
        if band is 'B5':
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)
        if band is 'MTL':
            moveToRaw(file, 'LST', date, city)
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)            

Masked and reprojected TIF saved as Unprocessed\clipped_LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF


NameError: name 'getLongitudeLatitudeOfTif' is not defined